# Policy Decision Map

Визуализация политики агента (правого paddle) в пространстве состояний.

**Осп графика:**
- Ось X → позиция платформы агента `py`
- Ось Y → позиция мяча по вертикали `by`

**Панель 1 — Preference map:** `P(вверх) − P(вниз)`. Красный = агент хочет идти вверх, синий = вниз, белый = нейтрально/stay.

**Панель 2 — Certainty map:** `max(P(вверх), P(вниз), P(stay))`. Насколько уверена модель + символы доминирующего действия.

**Панель 3 — Velocity arrow:** текущий вектор скорости мяча.

Используйте слайдеры для интерактивного изменения параметров.

In [1]:
import sys
import os

# Добавляем корень проекта в путь
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.agent.actor_critic import ActorCriticAgent
from run.config import ActorCriticConfig, EnvConfig

print("Imports OK")

Imports OK


In [2]:
# ---------------------------------------------------------------------------
# Загрузка агента
# ---------------------------------------------------------------------------
CHECKPOINT = os.path.join(PROJECT_ROOT, "artifacts", "actor_critic_model.pt")

cfg = ActorCriticConfig()
agent = ActorCriticAgent(
    state_dim=cfg.state_dim,
    action_dim=cfg.action_dim,
    hidden_dim=cfg.hidden_dim,
    device="cpu",
)
agent.load(CHECKPOINT)
agent.network.eval()
print(f"Checkpoint loaded: {CHECKPOINT}")

Checkpoint loaded: /Users/vasilij/Documents/RL_proj4/HorizontalPong/artifacts/actor_critic_model.pt


In [3]:
# ---------------------------------------------------------------------------
# Константы окружения
# ---------------------------------------------------------------------------
W   = EnvConfig.width           # ширина поля
H   = EnvConfig.height          # высота поля
PH  = EnvConfig.paddle_height   # высота платформы
MAX_VX = EnvConfig.max_ball_speed_x
MAX_VY = EnvConfig.max_ball_speed_y

PY_MIN = PH // 2                # минимальная позиция центра платформы
PY_MAX = H - 1 - PH // 2       # максимальная позиция

# Все допустимые позиции мяча и платформы
BALL_Y_RANGE  = np.arange(0, H)           # 0 .. H-1
AGENT_Y_RANGE = np.arange(PY_MIN, PY_MAX + 1)  # PH//2 .. H-1-PH//2

print(f"Field: {W}x{H}  |  Agent py: [{PY_MIN}, {PY_MAX}]  |  Ball by: [0, {H-1}]")
print(f"Grid size: {len(BALL_Y_RANGE)} x {len(AGENT_Y_RANGE)} = {len(BALL_Y_RANGE)*len(AGENT_Y_RANGE)} states")

Field: 86x64  |  Agent py: [6, 57]  |  Ball by: [0, 63]
Grid size: 64 x 52 = 3328 states


In [4]:
# ---------------------------------------------------------------------------
# Вычисление карты политики
# ---------------------------------------------------------------------------
@torch.no_grad()
def compute_policy_map(bx: int, vx: int, vy: int) -> dict:
    """
    Для каждой пары (ball_y, agent_y) вычисляем вероятности действий.

    Returns dict с массивами shape (len(BALL_Y_RANGE), len(AGENT_Y_RANGE)):
        prob_up, prob_down, prob_stay — вероятности действий
        preference   — P(up) - P(down)
        certainty    — max(P(up), P(down), P(stay))
        dominant     — 0/1/2 (up/down/stay)
    """
    n_by = len(BALL_Y_RANGE)
    n_py = len(AGENT_Y_RANGE)

    # Формируем батч всех (by, py) комбинаций
    by_grid, py_grid = np.meshgrid(BALL_Y_RANGE, AGENT_Y_RANGE, indexing="ij")
    by_flat = by_grid.ravel().astype(np.float32)
    py_flat = py_grid.ravel().astype(np.float32)

    bx_norm = bx / float(W)
    vx_norm = vx / float(max(1, MAX_VX))
    vy_norm = vy / float(max(1, MAX_VY))

    states = np.stack([
        np.full(len(by_flat), bx_norm),
        by_flat / float(H),
        np.full(len(by_flat), vx_norm),
        np.full(len(by_flat), vy_norm),
        py_flat / float(H),
    ], axis=1).astype(np.float32)

    states_t = torch.FloatTensor(states)
    logits = agent.network.get_action(states_t)          # (N, 3)
    probs  = torch.softmax(logits, dim=-1).numpy()        # (N, 3)

    prob_up   = probs[:, 0].reshape(n_by, n_py)
    prob_down = probs[:, 1].reshape(n_by, n_py)
    prob_stay = probs[:, 2].reshape(n_by, n_py)

    return {
        "prob_up":    prob_up,
        "prob_down":  prob_down,
        "prob_stay":  prob_stay,
        "preference": prob_up - prob_down,
        "certainty":  np.max(probs.reshape(n_by, n_py, 3), axis=-1),
        "dominant":   np.argmax(probs.reshape(n_by, n_py, 3), axis=-1),
    }

In [5]:
# ---------------------------------------------------------------------------
# Функция отрисовки
# ---------------------------------------------------------------------------
ACTION_SYMBOLS = {0: "↑", 1: "↓", 2: "●"}
ACTION_COLORS  = {0: "#d62728", 1: "#1f77b4", 2: "#2ca02c"}

def draw_velocity_arrow(ax, vx, vy):
    """Рисует вектор скорости мяча на отдельной оси."""
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.axhline(0, color="#cccccc", lw=0.8)
    ax.axvline(0, color="#cccccc", lw=0.8)
    ax.set_xticks(range(-MAX_VX, MAX_VX + 1))
    ax.set_yticks(range(-MAX_VY, MAX_VY + 1))
    ax.tick_params(labelsize=7)
    ax.set_title("Velocity\nvector", fontsize=9)

    # Сетка допустимых скоростей
    for vxi in range(-MAX_VX, MAX_VX + 1):
        for vyi in range(-MAX_VY, MAX_VY + 1):
            ax.plot(vxi, -vyi, "o", color="#dddddd", ms=4, zorder=1)

    # Текущий вектор (vy инвертируем: в матрице y растёт вниз, в графике — вверх)
    ax.annotate("", xy=(vx, -vy), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=2.0))
    ax.plot(vx, -vy, "o", color="#e74c3c", ms=7, zorder=3)
    ax.set_xlabel("vx", fontsize=8)
    ax.set_ylabel("-vy", fontsize=8)


def render_decision_map(bx, vx, vy):
    data = compute_policy_map(bx, vx, vy)

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5.5),
        gridspec_kw={"width_ratios": [5, 5, 2], "wspace": 0.35},
    )
    ax_pref, ax_cert, ax_vel = axes
    fig.suptitle(
        f"Policy Decision Map  |  ball x={bx}  |  vx={vx:+d}  vy={vy:+d}",
        fontsize=13, fontweight="bold", y=1.01,
    )

    # Use standard plot orientation (y increases upward) to avoid
    # perceived up/down inversion in the heatmap interpretation.
    extent = [AGENT_Y_RANGE[0] - 0.5, AGENT_Y_RANGE[-1] + 0.5,
              BALL_Y_RANGE[0]  - 0.5, BALL_Y_RANGE[-1]  + 0.5]

    # ------------------------------------------------------------------
    # Панель 1: Preference map  P(up) - P(down)
    # ------------------------------------------------------------------
    im1 = ax_pref.imshow(
        data["preference"],
        cmap="RdBu", vmin=-1, vmax=1,
        aspect="auto", extent=extent, interpolation="nearest", origin="lower",
    )
    plt.colorbar(im1, ax=ax_pref, label="P(up) − P(down)", fraction=0.035)

    # Нулевая граница
    ax_pref.contour(
        AGENT_Y_RANGE, BALL_Y_RANGE, data["preference"],
        levels=[0], colors="white", linewidths=1.2, linestyles="--",
    )
    # Диагональ: по = by (агент совпадает с мячом)
    diag = np.intersect1d(AGENT_Y_RANGE, BALL_Y_RANGE)
    if len(diag) > 1:
        ax_pref.plot(diag, diag, color="yellow", lw=1.5, ls="-", label="py = by", zorder=5)
        ax_pref.legend(fontsize=8, loc="upper left")

    # Вертикаль — текущий bx (только как подсказка)
    ax_pref.set_xlabel("Agent y (py)", fontsize=10)
    ax_pref.set_ylabel("Ball y (by)", fontsize=10)
    ax_pref.set_title("Preference map\nP(↑) − P(↓)", fontsize=10)

    # ------------------------------------------------------------------
    # Панель 2: Certainty + dominant action symbols
    # ------------------------------------------------------------------
    im2 = ax_cert.imshow(
        data["certainty"],
        cmap="Greens", vmin=0.33, vmax=1.0,
        aspect="auto", extent=extent, interpolation="nearest", origin="lower",
    )
    plt.colorbar(im2, ax=ax_cert, label="max P(action)", fraction=0.035)

    # Символы доминирующего действия (прореживаем сетку для читаемости)
    step_by = max(1, len(BALL_Y_RANGE)  // 16)
    step_py = max(1, len(AGENT_Y_RANGE) // 16)
    for i, by_val in enumerate(BALL_Y_RANGE[::step_by]):
        for j, py_val in enumerate(AGENT_Y_RANGE[::step_py]):
            ii = i * step_by
            jj = j * step_py
            act = data["dominant"][ii, jj]
            ax_cert.text(
                py_val, by_val, ACTION_SYMBOLS[act],
                ha="center", va="center",
                fontsize=8, color=ACTION_COLORS[act], fontweight="bold",
            )

    # Диагональ
    if len(diag) > 1:
        ax_cert.plot(diag, diag, color="yellow", lw=1.5, ls="-", label="py = by", zorder=5)
        ax_cert.legend(fontsize=8, loc="upper left")

    ax_cert.set_xlabel("Agent y (py)", fontsize=10)
    ax_cert.set_ylabel("Ball y (by)", fontsize=10)
    ax_cert.set_title("Certainty map\nmax P(action)  +  dominant action", fontsize=10)

    # Легенда действий
    legend_patches = [
        mpatches.Patch(color=ACTION_COLORS[0], label="↑ Up"),
        mpatches.Patch(color=ACTION_COLORS[1], label="↓ Down"),
        mpatches.Patch(color=ACTION_COLORS[2], label="● Stay"),
    ]
    ax_cert.legend(handles=legend_patches, fontsize=8, loc="lower right")

    # ------------------------------------------------------------------
    # Панель 3: Velocity arrow
    # ------------------------------------------------------------------
    draw_velocity_arrow(ax_vel, vx, vy)

    plt.tight_layout()
    plt.show()

In [6]:
# ---------------------------------------------------------------------------
# Интерактивные виджеты
# ---------------------------------------------------------------------------

# vx: мяч движется к агенту (положительный), от агента (отрицательный), не 0
vx_options = [(f"vx = {v:+d}", v) for v in range(-MAX_VX, MAX_VX + 1) if v != 0]
vy_options = [(f"vy = {v:+d}", v) for v in range(-MAX_VY, MAX_VY + 1)]

w_bx = widgets.IntSlider(
    value=W // 2, min=1, max=W - 2, step=1,
    description="ball x (d):",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)
w_vx = widgets.SelectionSlider(
    options=vx_options,
    value=1,
    description="vx:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)
w_vy = widgets.IntSlider(
    value=0, min=-MAX_VY, max=MAX_VY, step=1,
    description="vy:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)

out = widgets.Output()

def on_change(_change=None):
    with out:
        clear_output(wait=True)
        render_decision_map(w_bx.value, w_vx.value, w_vy.value)

w_bx.observe(on_change, names="value")
w_vx.observe(on_change, names="value")
w_vy.observe(on_change, names="value")

controls = widgets.VBox([
    widgets.HTML("<b>Параметры состояния</b>"),
    w_bx, w_vx, w_vy,
])

display(controls, out)
on_change()  # первоначальная отрисовка

Output()